# `Business Case`

## `Requerimientos Funcionales`

* `Añadir` productos al inventario, incluyendo nombre, cantidad (enteros), precio unitario, valor total del inventario.

* `Eliminar` productos del inventario. Evitar el ofrecimiento de productos que ya no son ofrecidos por la organización.

* `Consultar` el estado actual de inventario. Medir la capacidad de oferta, puntos de reabastecimiento, etc.

* `Exportar` informe en formato PDF.

## `Requerimientos Técnicos`

* `Normalización` del nombre del producto (`title()`, `strip()`).

* `Separación` visual con caracteres (`"-" * 30`) para un diseño más organizado.

* `Menú` interactivo para que el usuario elija acciones (`while()`).

* `Representación` del inventario y productos con estructuras de datos (e.g. `list`, `dict`…).

* `Implementación` de la librería `fpdf`.  Para la exportación del informe en formato `PDF`.

* `Persistir` los datos en una Base de Datos con `SQLite`.

# 1. Crear `DB Inventario` y `tabla productos`

# 1.1 Creación y conexión a la DB

In [2]:
import sqlite3

while True:
  try:
    conn = sqlite3.connect(database="inventario_dev.db")
    cur = conn.cursor()
    print("Conexión a la DB lograda con éxito")
    break
  except Exception as e:
    print(f"Error al conectarse a la DB: {e}")

Conexión a la DB lograda con éxito


# 1.2 Crear la tabla `productos` en la DB inventario

In [4]:
try:
  cur.execute(
    """
    CREATE TABLE productos (
      id INTEGER PRIMARY KEY AUTOINCREMENT,
      fecha_creacion DATE NOT NULL DEFAULT(DATE('now')),
      nombre VARCHAR(100) NOT NULL,
      cantidad INTEGER NOT NULL,
      precio_unitario DECIMAL NOT NULL,
      valor_total DECIMAL
    );
    """
  )
  print("Tabla creado con éxito")
except Exception as e:
  print(f"Error al crear la tabla {e}")

Tabla creado con éxito


# 2. Diseño del CRUD

# 2.1 CREATE (un solo producto)

In [8]:
while True:
  try:
    producto = input("Ingrese el nombre del producto: ").strip().title()
    if not producto.isalpha():
      raise ValueError(f"El nombre del producto no puede contener núermos. Nombre de producto ingresado {producto}")
  except Exception as e:
    print(f"ERROR, {e}")
  else:
    break
  
while True:
  try:
    cantidad = input("Ingrese la cantidad inicial en el inventario")
    if not cantidad.isdigit():
      raise ValueError(f"La cantidad de inventario inicial debe ser un número entero positivo. Cantidad ingresada: {cantidad}")
  except Exception as e:
    print(f"ERROR, {e}")
  else:
    cantidad = int(cantidad)
    break
  
while True:
  try:
    precio_unitario = input("Ingrese el precio unitario del producto")
    if not (precio_unitario.replace(".", "").isdigit() or precio_unitario.replace(",", ".").isdigit()):
      raise ValueError(f"El precio unitario debe ser un número positivo (ej: 10.50s). Precio ingresado: {precio_unitario}")
  except Exception as e:
    print(f"ERROR, {e}")
  else:
    if precio_unitario.__contains__(","):
      precio_unitario = float(precio_unitario.replace(",", "."))
    else:
      precio_unitario = float(precio_unitario)
    break

producto_dict = {
  "nombre": producto,
  "cantidad": cantidad,
  "precio_unitario": precio_unitario,
  "valor_total": precio_unitario * cantidad
}

In [9]:
producto_dict

{'nombre': 'Celular',
 'cantidad': 10,
 'precio_unitario': 123.0,
 'valor_total': 1230.0}

In [10]:
try:
  cur.execute(
  """
    INSERT INTO 
      productos (nombre, cantidad, precio_unitario, valor_total)
    VALUES
      (?,?,?,?)
  """,
  (
    producto_dict["nombre"], 
    producto_dict["cantidad"], 
    producto_dict["precio_unitario"], 
    producto_dict["valor_total"])
  )
  conn.commit()
  print("El producto fue agregado con éxito al inventario")
except Exception as e:
  print(f"Error al crear el producto. {e}")

El producto fue agregado con éxito al inventario


# 2.2 READ (todos los productos)

In [3]:
datos = cur.execute(
  """
    SELECT * FROM productos
  """
).fetchall()
print(datos)

[(1, '2026-10-01', 'Televisor', 3, 3.45, 10.350000000000001), (2, '2026-10-01', 'Celular', 10, 123, 1230)]


In [5]:
for dato in datos:
  print(f"ID                     : {dato[0]}")
  print(f"Fecha                  : {dato[1]}")
  print(f"Producto               : {dato[2]}")
  print(f"Cantidad               : {dato[3]}")
  print(f"Precio unitario        : {dato[4]}")
  print(f"Valor total            : {dato[5]}")
  print("-" * 30)

ID                     : 1
Fecha                  : 2026-10-01
Producto               : Televisor
Cantidad               : 3
Precio unitario        : 3.45
Valor total            : 10.350000000000001
------------------------------
ID                     : 2
Fecha                  : 2026-10-01
Producto               : Celular
Cantidad               : 10
Precio unitario        : 123
Valor total            : 1230
------------------------------


# 2.3 READ (un solo producto)

In [6]:
dato = cur.execute(
  """
    SELECT * FROM productos WHERE id = 2
  """
).fetchone()
print(dato)

(2, '2026-10-01', 'Celular', 10, 123, 1230)


In [7]:
print(f"ID                     : {dato[0]}")
print(f"Fecha                  : {dato[1]}")
print(f"Producto               : {dato[2]}")
print(f"Cantidad               : {dato[3]}")
print(f"Precio unitario        : {dato[4]}")
print(f"Valor total            : {dato[5]}")
print("-" * 30)

ID                     : 2
Fecha                  : 2026-10-01
Producto               : Celular
Cantidad               : 10
Precio unitario        : 123
Valor total            : 1230
------------------------------


# 2.4 UPDATE

In [8]:
try:
  cur.execute(
    """
      UPDATE productos SET fecha_creacion = '2026-09-25' WHERE id = 2;
    """
  )
  conn.commit()
  print("Producto actualizado exitosamente")
except Exception as e:
  print(f"Error al actualizar el producto {e}")

Producto actualizado exitosamente


# 2.5 DELETE

In [9]:
try:
  cur.execute(
    """
      DELETE FROM productos WHERE id = 2;
    """
  )
  conn.commit()
  print("Producto eliminado exitosamente")
except Exception as e:
  print(f"Error al eliminar el producto {e}")

Producto eliminado exitosamente


# 3. Generar informe en PDF